In [139]:
import pandas as pd
import numpy as np
from scipy.io import loadmat
from pathlib import Path

In [140]:
data_folder = Path('../Data/')
raw_data_folder = Path('../Data/raw_data/')

In [141]:
files = raw_data_folder.glob("*.mat")
print(len(list(files)))
for file in raw_data_folder.glob("*.mat"):
    data = loadmat(file)

    print(file.name)
    print(data.keys())
    print(data['Data'])
    break

480
Arithmetic_sub_6_trial2.mat
dict_keys(['__header__', '__version__', '__globals__', 'Data'])
[[  50.32077746   51.7268748    57.83747759 ...   64.05595567
    67.0437884    65.87162413]
 [-151.53716283 -147.9897562  -141.75262434 ...  -41.78080769
   -47.7988956   -56.65061251]
 [-239.09247126 -239.3364102  -230.30001949 ... -103.13494497
  -111.3604676  -112.24444823]
 ...
 [  50.960909     50.83676317   58.734015   ...   56.03909857
    52.17734349   50.33941157]
 [   6.917256      5.59297689   13.096351   ...   46.06338729
    33.86831134   30.07264329]
 [-364.243526   -359.70764797 -350.338643   ... -151.85826307
  -167.62453659 -181.50147457]]


# Reading Data

In [142]:
scales = pd.read_excel(data_folder / 'scales.xls', header=[0, 1])

In [143]:
scales.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   (Subject No., Unnamed: 0_level_1)  40 non-null     int64
 1   (Trial_1, Maths)                   40 non-null     int64
 2   (Trial_1, Symmetry)                40 non-null     int64
 3   (Trial_1, Stroop)                  40 non-null     int64
 4   (Trial_2, Maths)                   40 non-null     int64
 5   (Trial_2, Symmetry)                40 non-null     int64
 6   (Trial_2, Stroop)                  40 non-null     int64
 7   (Trial_3, Maths)                   40 non-null     int64
 8   (Trial_3, Symmetry)                40 non-null     int64
 9   (Trial_3, Stroop)                  40 non-null     int64
dtypes: int64(10)
memory usage: 3.3 KB


In [144]:
scales.head()

Subject No. Trial_1                 Trial_2                 Trial_3  \
  Unnamed: 0_level_1   Maths Symmetry Stroop   Maths Symmetry Stroop   Maths   
0                  1       6        3      3       7        5      2       4   
1                  2       3        4      5       3        4      4       7   
2                  3       5        3      4       3        5      5       8   
3                  4       5        3      4       3        5      2       7   
4                  5       6        6      6       5        3      2       5   

                   
  Symmetry Stroop  
0        7      4  
1        5      3  
2        7      5  
3        5      5  
4        7      3

` It is complex so I make it a simple lookup table`

In [145]:
scales.columns = [
    f'{trial}_{task}'
    if trial != 'Subject No.'
    else "Subject_No"
    for trial, task in scales.columns
]
scales.set_index('Subject_No', inplace=True)

In [146]:
scales.head()

,Trial_1_Maths,Trial_1_Symmetry,Trial_1_Stroop,Trial_2_Maths,Trial_2_Symmetry,Trial_2_Stroop,Trial_3_Maths,Trial_3_Symmetry,Trial_3_Stroop
Subject_No,,,,,,,,,
1,6,3,3,7,5,2,4,7,4
2,3,4,5,3,4,4,7,5,3
3,5,3,4,3,5,5,8,7,5
4,5,3,4,3,5,2,7,5,5
5,6,6,6,5,3,2,5,7,3


In [147]:
subject1 = scales.loc[1]
subject1

Trial_1_Maths       6
Trial_1_Symmetry    3
Trial_1_Stroop      3
Trial_2_Maths       7
Trial_2_Symmetry    5
Trial_2_Stroop      2
Trial_3_Maths       4
Trial_3_Symmetry    7
Trial_3_Stroop      4
Name: 1, dtype: int64

# Create A Master DataFrame

In [170]:
data = []
for file in raw_data_folder.glob("*.mat"):
    name = file.name
    if name.startswith('Mirror'):
        name = name.replace('_image', '')

    # print(name)
    kind = name.split('_')[0]
    subject = int(name.split('_')[2])
    trial = int(name.split('_')[3][5:6])

    sample = {
        "subject": subject,
        "trial": trial,
        "kind": kind
    }

    data.append(sample)


df = pd.DataFrame(data)

In [171]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   subject  480 non-null    int64
 1   trial    480 non-null    int64
 2   kind     480 non-null    str  
dtypes: int64(2), str(1)
memory usage: 11.4 KB


In [172]:
df.head()

,subject,trial,kind
0,6,2,Arithmetic
1,10,1,Relax
2,36,3,Relax
3,27,1,Relax
4,17,2,Stroop


In [173]:
df.sort_values(by=['subject', 'trial'], inplace=True)
df.head(5)

,subject,trial,kind
149,1,1,Relax
161,1,1,Mirror
214,1,1,Arithmetic
329,1,1,Stroop
127,1,2,Mirror


In [174]:
df['kind'].value_counts()

kind
Relax         120
Mirror        120
Arithmetic    120
Stroop        120
Name: count, dtype: int64

In [180]:
def extract_stress_level(scores: pd.DataFrame, subject: int, trial: int, kind: str) -> int:
    subject = scores.loc[subject]
    mapping = {
        'Arithmetic': 'Maths',
        'Mirror': 'Symmetry',
        'Stroop': 'Stroop'
    }
    if kind not in mapping:
        return 0
    
    kind = mapping[kind]
    column_name = f'Trial_{trial}_{kind}'
    return subject[column_name]

In [181]:
df['stress_level'] = df.apply(
    lambda row: extract_stress_level(
        scales,
        row['subject'],
        row['trial'],
        row['kind']
    ),
    axis=1
)

In [182]:
df.head()

,subject,trial,kind,stress_level
149,1,1,Relax,0
161,1,1,Mirror,3
214,1,1,Arithmetic,6
329,1,1,Stroop,3
127,1,2,Mirror,5
